# Rust Performance Backend

This notebook demonstrates the Rust backend for high-performance operations.

**Key feature:** The Python `Scale` class now automatically uses the Rust backend when available. All arithmetic operations (`*`, `/`, `**`) are delegated to Rust for maximum performance.

In [ ]:
# Note: The Rust backend must be built with `maturin develop`
try:
    from physities._physities_core import PhysicalScale
    HAS_RUST = True
    print("Rust backend available!")
except ImportError:
    HAS_RUST = False
    print("Rust backend not available. Build with `maturin develop`.")

## Automatic Rust Acceleration

The Python `Scale` class automatically uses the Rust backend for all operations when available:

In [ ]:
from physities.src.unit import Meter, Second, Kilometer, Hour
from physities.src.scale import Scale

# Check if Rust backend is being used
print(f"Rust backend available: {Scale.has_rust_backend()}")

# All these operations automatically use Rust when available:
print("\n# Scale operations (automatically accelerated by Rust):")

# Division
Ms = Meter / Second
print(f"Meter / Second: L={Ms.scale.dimension.length}, T={Ms.scale.dimension.time}")

# Power
m2 = Meter ** 2
print(f"Meter^2: L={m2.scale.dimension.length}")

# Scalar multiplication
km_scale = Meter.scale * 1000
print(f"Meter * 1000 conversion factor: {km_scale.conversion_factor}")

# Unit conversion (uses Rust-accelerated Scale operations)
print("\n# Unit conversion example:")
v_kmh = (Kilometer / Hour)(360)  # 360 km/h
v_ms = v_kmh.convert(Ms)
print(f"360 km/h = {v_ms.value:.2f} m/s")

## Using PhysicalScale Directly

The Rust `PhysicalScale` provides high-performance scale operations.

In [ ]:
if HAS_RUST:
    # Create a velocity scale (m/s)
    # Dimensions: (length, mass, temp, time, amount, current, luminosity)
    velocity_scale = PhysicalScale.from_components(
        (1.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0),  # dimension exponents
        (1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0),   # conversion factors
        1.0  # rescale value
    )
    
    print(f"Length exponent: {velocity_scale.length}")
    print(f"Time exponent: {velocity_scale.time}")
    print(f"Conversion factor: {velocity_scale.conversion_factor}")

## Scale Operations

In [ ]:
if HAS_RUST:
    # Power operation
    velocity_squared = velocity_scale.power(2)
    print(f"Velocity squared dimensions: L={velocity_squared.length}, T={velocity_squared.time}")
    
    # Multiply scales
    mass_scale = PhysicalScale.from_components(
        (0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0),
        (1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0),
        1.0
    )
    
    # Kinetic energy scale: m * v^2
    energy_scale = mass_scale.multiply(velocity_squared)
    print(f"Energy dimensions: L={energy_scale.length}, M={energy_scale.mass}, T={energy_scale.time}")

## Serialization

In [ ]:
if HAS_RUST:
    # JSON serialization
    json_str = velocity_scale.to_json()
    print(f"JSON: {json_str}")
    
    # Reconstruct from JSON
    reconstructed = PhysicalScale.from_json(json_str)
    print(f"Reconstructed: L={reconstructed.length}, T={reconstructed.time}")
    
    # Int64 encoding for compact storage
    encoded = velocity_scale.to_dimension_int64()
    print(f"Int64 encoded: {encoded}")

## NumPy Interop

In [ ]:
if HAS_RUST:
    try:
        import numpy as np
        
        # Get the underlying array
        arr = velocity_scale.as_numpy()
        print(f"NumPy array shape: {arr.shape}")
        print(f"Array contents: {arr}")
    except ImportError:
        print("NumPy not available")